<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-04-image-and-multimodal-generation/lesson-4.4-vit-and-clip/notebooks/exercises-4_4.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 4.4: Vision Transformers (ViT) & CLIP

*Module 4 · Image, Vision & Multimodal AI*

> ViT treats images as patch sequences and feeds them to Transformers. CLIP puts images and text in the same embedding space. Together they power zero-shot classification, image search, and the text encoder inside Stable Diffusion. This lesson codes both.

`ViT` · `CLIP` · `Zero-Shot` · `Embeddings` · `60 min`

---

## About this notebook

This notebook contains the **6 runnable code examples** from the published lesson `lesson-4.4.html`. Each block is reproduced verbatim — every `#` comment from the source is preserved — and is preceded by a short markdown header that names the step, the file, and what the block demonstrates.

**How to use it:**

1. Run the **Setup** cells once (install + API keys).
2. Step through the lesson cells in order — each is independent enough to inspect on its own.
3. Read the *What just happened?* recap that follows each block (where the lesson provides one).


## Contents

1. Step 1 — Vision Transformer — Images as Token Sequences — `01_vit_patches.py`
2. Step 2 — CLIP — Images and Text in the Same Space — `02_clip_similarity.py`
3. Step 3 — Zero-Shot Classification — No Labels Needed — `03_zero_shot.py`
4. Step 4 — CLIP for Image Search — Find Images with Text — `04_clip_search.py`
5. Step 5 — Contrastive Learning — How CLIP Actually Trains — `contrastive_learning.py`
6. Step 9 — Production: Vector Search, Content Moderation & timm — `clip_production.py`


## Setup

Run these cells once per environment. Skip any step you've already done.


In [ ]:
# Install Python dependencies used by this lesson.
# The -q flag keeps output quiet; remove it if you want full pip logs.
!pip install -q openai transformers torch numpy faiss-cpu requests pillow


## Lesson code

Each section below corresponds to one code block from the published lesson.


### Step 1 · Vision Transformer — Images as Token Sequences

Split image into patches, flatten, add position embeddings, feed to Transformer.

**`01_vit_patches.py`** · _Block 1 of 6_

ViT — Patching an Image Like Tokenizing Text


In [ ]:
# ViT — Patching an Image Like Tokenizing Text
import torch, torch.nn as nn

class PatchEmbedding(nn.Module):
    def __init__(self, img_size=224, patch_size=16, d_model=768):
        super().__init__()
        self.proj = nn.Conv2d(3, d_model, kernel_size=patch_size, stride=patch_size)
    def forward(self, x):
        return self.proj(x).flatten(2).transpose(1,2)

class SimpleViT(nn.Module):
    def __init__(self, img_size=224, patch_size=16, d=768, layers=12, heads=12, classes=1000):
        super().__init__()
        n = (img_size//patch_size)**2
        self.patch = PatchEmbedding(img_size, patch_size, d)
        self.cls = nn.Parameter(torch.randn(1,1,d))
        self.pos = nn.Parameter(torch.randn(1,n+1,d))
        self.enc = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d,heads,d*4,batch_first=True), layers)
        self.head = nn.Linear(d, classes)
    def forward(self, x):
        t = torch.cat([self.cls.expand(x.size(0),-1,-1), self.patch(x)], dim=1)
        return self.head(self.enc(t + self.pos)[:,0])

vit = SimpleViT()
out = vit(torch.randn(1,3,224,224))
print(f"ViT-B/16: 224×224 → 196 patches → {out.shape[1]} classes")
print(f"Params: {sum(p.numel() for p in vit.parameters()):,}")
print(f"Same Transformer as BERT/GPT — patches instead of words!")


> **What just happened?** A 224×224 image was chopped into 196 patches (16×16 each), projected to 768-dim vectors, prepended with [CLS], and fed through 12 Transformer layers. Exact same architecture as BERT — the only difference is patches instead of words.


### Step 2 · CLIP — Images and Text in the Same Space

Trained on 400M image-text pairs. Now photos and descriptions live together as vectors.

**`02_clip_similarity.py`** · _Block 2 of 6_

CLIP — Image-Text Similarity


In [ ]:
# CLIP — Image-Text Similarity
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
import requests, torch

model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

url = "https://upload.wikimedia.org/wikipedia/commons/thumb/4/4d/Cat_November_2010-1a.jpg/220px-Cat_November_2010-1a.jpg"
image = Image.open(requests.get(url, stream=True).raw)

candidates = ["a photo of a cat", "a photo of a dog", "a photo of a car",
              "a cute kitten", "a landscape", "an orange tabby cat"]

inputs = processor(text=candidates, images=image, return_tensors="pt", padding=True)
with torch.no_grad():
    probs = model(**inputs).logits_per_image[0].softmax(dim=-1)

print("CLIP Image-Text Similarity:
")
for t, p in sorted(zip(candidates, probs.tolist()), key=lambda x:-x[1]):
    print(f"  {p:.3f} {'█'*int(p*40)} '{t}'")
print(f"
ZERO-SHOT: no training on this image!")


> **What just happened?** CLIP encoded the cat photo AND all 6 texts into the same 512-dim space. “orange tabby cat” scored highest. Zero-shot: classify any image into any categories without fine-tuning. Just describe the categories in text.


### Step 3 · Zero-Shot Classification — No Labels Needed

Classify images into ANY text-defined categories. No training required.

**`03_zero_shot.py`** · _Block 3 of 6_

Zero-Shot Image Classification with CLIP


In [ ]:
# Zero-Shot Image Classification with CLIP
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
import torch, requests

model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

def classify(image, categories):
    prompts = [f"a photo of {c}" for c in categories]
    inputs = processor(text=prompts, images=image, return_tensors="pt", padding=True)
    with torch.no_grad():
        probs = model(**inputs).logits_per_image[0].softmax(dim=-1)
    return {c: p.item() for c, p in zip(categories, probs)}

url = "https://upload.wikimedia.org/wikipedia/commons/thumb/4/4d/Cat_November_2010-1a.jpg/220px-Cat_November_2010-1a.jpg"
img = Image.open(requests.get(url, stream=True).raw)

# Same image, two DIFFERENT classification schemes
print("By species:", classify(img, ["cat","dog","bird","fish"]))
print("By mood:", classify(img, ["happy animal","sleepy animal","curious animal"]))
print("
Same model, different categories. No retraining. That is zero-shot.")


### Step 4 · CLIP for Image Search — Find Images with Text

Embed a gallery, search with natural language. Same principle as Google Photos.

**`04_clip_search.py`** · _Block 4 of 6_

CLIP Image Search Engine


In [ ]:
# CLIP Image Search Engine
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
import torch, numpy as np

model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

class CLIPSearch:
    def __init__(self): self.embs, self.names = [], []
    def index(self, image, name):
        inputs = processor(images=image, return_tensors="pt")
        with torch.no_grad(): e = model.get_image_features(**inputs)
        self.embs.append((e/e.norm())[0].numpy()); self.names.append(name)
    def search(self, query, k=3):
        inputs = processor(text=[query], return_tensors="pt")
        with torch.no_grad(): e = model.get_text_features(**inputs)
        e = (e/e.norm())[0].numpy()
        scores = [(n, np.dot(e, ie)) for n, ie in zip(self.names, self.embs)]
        return sorted(scores, key=lambda x:-x[1])[:k]

s = CLIPSearch()
for name, rgb in [("red",(255,0,0)),("green",(0,180,0)),("blue",(0,100,255)),("yellow",(255,230,0))]:
    s.index(Image.new("RGB",(224,224),rgb), name)

for q in ["ocean waves","fire","forest","sunshine"]:
    print(f"  '{q}' → {s.search(q, 2)}")
print("\nThis is how Google Photos search works at scale.")


### Step 5 · Contrastive Learning — How CLIP Actually Trains

400M pairs. Batch size 32,768. InfoNCE loss. No labels needed.

**`contrastive_learning.py`** · _Block 5 of 6_

═══════ CLIP CONTRASTIVE TRAINING (simplified) ═══════


In [ ]:
# ═══════ CLIP CONTRASTIVE TRAINING (simplified) ═══════
import torch
import torch.nn.functional as F

def clip_loss(image_embeddings, text_embeddings, temperature=0.07):
    # L2 normalize
    img_emb = F.normalize(image_embeddings, dim=-1)
    txt_emb = F.normalize(text_embeddings, dim=-1)
    
    # Compute N×N similarity matrix
    logits = (img_emb @ txt_emb.T) / temperature  # [N, N]
    
    # Labels: diagonal = correct pairs (0,1,2,...,N-1)
    labels = torch.arange(len(logits), device=logits.device)
    
    # Symmetric cross-entropy: image→text AND text→image
    loss_i2t = F.cross_entropy(logits, labels)      # Each image picks correct text
    loss_t2i = F.cross_entropy(logits.T, labels)     # Each text picks correct image
    
    return (loss_i2t + loss_t2i) / 2

# With batch=32768: each image contrasts against 32,767 negatives
# Temperature τ is LEARNED via gradient descent (not fixed)
# Training data: 400M image-text pairs (WebImageText, never released)


> **What just happened?** Contrastive learning in 30 seconds: Take a batch of N image-text pairs. Compute all N² possible similarities. Correct pairs (diagonal) should score high. Wrong pairs (off-diagonal) should score low. Cross-entropy loss on each row and column. The internet is the label — image alt-text from 400M web pages provides free supervision. Large batches (32K) provide hard negatives. The learned temperature τ controls how confident the model must be.


### Step 9 · Production: Vector Search, Content Moderation & timm

FAISS + CLIP for image search. Zero-shot moderation. timm for pre-trained ViTs.

**`clip_production.py`** · _Block 6 of 6_

═══════ CLIP + FAISS for image search at scale ═══════


In [ ]:
# ═══════ CLIP + FAISS for image search at scale ═══════
import faiss, numpy as np

# Build index (offline)
dimension = 768  # ViT-L/14 output dimension
index = faiss.IndexFlatIP(dimension)  # Inner product = cosine for L2-normalized vectors
# embeddings = clip_model.get_image_features(images)  # batch encode
# embeddings = F.normalize(embeddings, dim=-1)  # ALWAYS L2-normalize!
# index.add(embeddings.numpy())

# Search (online): text → image retrieval
# query = clip_model.get_text_features(["red saree with gold border"])
# query = F.normalize(query, dim=-1)
# distances, indices = index.search(query.numpy(), k=10)

# ═══════ timm: largest pre-trained ViT collection ═══════
import timm

# Create pre-trained ViT with custom classes
model = timm.create_model('vit_base_patch16_224', pretrained=True, num_classes=10)

# Feature extraction (remove classification head)
model = timm.create_model('vit_base_patch16_224', pretrained=True, num_classes=0)
# features = model(tensor)  # Returns 768-dim embedding

# timm has 1000+ pre-trained models: DINOv2, DINOv3, MAE, etc.
# timm.list_models('vit_*', pretrained=True)  # List all ViTs

# Vector databases for CLIP at scale:
# FAISS: local library, prototyping (free)
# Qdrant: production <5M vectors (self-hosted)
# Pinecone: managed cloud (~$25/mo)
# Milvus: billion-scale (complex to deploy)
# Weaviate: hybrid vector+keyword (e-commerce)


> **What just happened?** CLIP + FAISS is the standard pattern for image search: encode images offline → FAISS index → search with text queries at ~1ms latency. Always L2-normalize embeddings — this makes inner product = cosine similarity. timm is the largest pre-trained ViT library (1000+ models including DINOv2/v3, MAE). For production: FAISS for prototyping, Qdrant for production, Weaviate for e-commerce hybrid search. An L4 GPU processes 500-1000 CLIP embeddings/second.


---

## Wrap-up

You've walked through all 6 code examples from this lesson. Re-run any cell to experiment — change a prompt, swap a model, raise the temperature. The published lesson page (linked at the top) has the surrounding narrative, quizzes, and practice exercises if you want to go deeper.
